In [1]:
import pandas as pd
import os

golden_folder = "golden_dataset"

data = {}

for file in os.listdir(golden_folder):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        data[name] = pd.read_csv(
            os.path.join(golden_folder, file)
        )

print("Golden dataset loaded successfully.")
print("Total tables:", len(data))

Golden dataset loaded successfully.
Total tables: 18


In [2]:
# Check targeting strategy over time

targeting = data["daily_targeting"].copy()

targeting["target_date"] = pd.to_datetime(
    targeting["target_date"]
)

targeting["month"] = (
    targeting["target_date"]
    .dt.to_period("M")
    .astype(str)
)

strategy_summary = (
    targeting.groupby(
        ["month", "campaign_id", "priority", "recommended_channel"]
    )
    .size()
    .reset_index(name="targeted_records")
)

strategy_summary

,month,campaign_id,priority,recommended_channel,targeted_records
0,2026-01,CMP0000001,1,FIELD,2
1,2026-01,CMP0000001,1,SMS,1
2,2026-01,CMP0000001,1,VOICE,1
3,2026-01,CMP0000001,2,FIELD,3
4,2026-01,CMP0000001,2,VOICE,3
...,...,...,...,...,...
25679,2026-08,CMP0000120,8,FIELD,1
25680,2026-08,CMP0000120,8,VOICE,1
25681,2026-08,CMP0000120,8,WHATSAPP,1
25682,2026-08,CMP0000120,9,SMS,1


In [3]:
# Summarize targeting strategy by month

strategy_monthly = (
    targeting.groupby(
        ["month", "campaign_id"]
    )
    .size()
    .reset_index(name="targeted_records")
)

strategy_monthly

,month,campaign_id,targeted_records
0,2026-01,CMP0000001,73
1,2026-01,CMP0000002,56
2,2026-01,CMP0000003,52
3,2026-01,CMP0000004,51
4,2026-01,CMP0000005,54
...,...,...,...
955,2026-08,CMP0000116,16
956,2026-08,CMP0000117,18
957,2026-08,CMP0000118,10
958,2026-08,CMP0000119,7


In [4]:
# Compare campaign usage by month

campaign_monthly = (
    targeting.groupby("month")["campaign_id"]
    .nunique()
    .reset_index(name="unique_campaigns")
)

campaign_monthly

,month,unique_campaigns
0,2026-01,120
1,2026-02,120
2,2026-03,120
3,2026-04,120
4,2026-05,120
5,2026-06,120
6,2026-07,120
7,2026-08,120


In [5]:
# Check priority and channel mix by month

strategy_mix = (
    targeting.groupby(
        ["month", "priority", "recommended_channel"]
    )
    .size()
    .reset_index(name="targeted_records")
)

strategy_mix

,month,priority,recommended_channel,targeted_records
0,2026-01,1,FIELD,137
1,2026-01,1,SMS,157
2,2026-01,1,VOICE,157
3,2026-01,1,WHATSAPP,163
4,2026-01,2,FIELD,172
...,...,...,...,...
315,2026-08,9,WHATSAPP,28
316,2026-08,10,FIELD,30
317,2026-08,10,SMS,41
318,2026-08,10,VOICE,31


In [6]:
# Calculate monthly targeting mix

strategy_mix_pct = (
    targeting.groupby(
        ["month", "recommended_channel"]
    )
    .size()
    .reset_index(name="records")
)

monthly_total = (
    targeting.groupby("month")
    .size()
    .reset_index(name="total_records")
)

strategy_mix_pct = strategy_mix_pct.merge(
    monthly_total,
    on="month"
)

strategy_mix_pct["percentage"] = (
    strategy_mix_pct["records"]
    / strategy_mix_pct["total_records"]
    * 100
).round(2)

strategy_mix_pct

,month,recommended_channel,records,total_records,percentage
0,2026-01,FIELD,1602,6369,25.15
1,2026-01,SMS,1608,6369,25.25
2,2026-01,VOICE,1543,6369,24.23
3,2026-01,WHATSAPP,1616,6369,25.37
4,2026-02,FIELD,1471,5709,25.77
5,2026-02,SMS,1379,5709,24.15
6,2026-02,VOICE,1429,5709,25.03
7,2026-02,WHATSAPP,1430,5709,25.05
8,2026-03,FIELD,1562,6290,24.83
9,2026-03,SMS,1616,6290,25.69


In [7]:
# Check priority mix by month

priority_mix = (
    targeting.groupby(["month", "priority"])
    .size()
    .reset_index(name="targeted_records")
)

monthly_total = (
    targeting.groupby("month")
    .size()
    .reset_index(name="total_records")
)

priority_mix = priority_mix.merge(
    monthly_total,
    on="month"
)

priority_mix["percentage"] = (
    priority_mix["targeted_records"]
    / priority_mix["total_records"]
    * 100
).round(2)

priority_mix

,month,priority,targeted_records,total_records,percentage
0,2026-01,1,614,6369,9.64
1,2026-01,2,644,6369,10.11
2,2026-01,3,636,6369,9.99
3,2026-01,4,639,6369,10.03
4,2026-01,5,608,6369,9.55
...,...,...,...,...,...
75,2026-08,6,141,1601,8.81
76,2026-08,7,151,1601,9.43
77,2026-08,8,188,1601,11.74
78,2026-08,9,163,1601,10.18


In [8]:
# Check targeting status by month

status_mix = (
    targeting.groupby(["month", "status"])
    .size()
    .reset_index(name="targeted_records")
)

monthly_total = (
    targeting.groupby("month")
    .size()
    .reset_index(name="total_records")
)

status_mix = status_mix.merge(
    monthly_total,
    on="month"
)

status_mix["percentage"] = (
    status_mix["targeted_records"]
    / status_mix["total_records"]
    * 100
).round(2)

status_mix

,month,status,targeted_records,total_records,percentage
0,2026-01,CONTACTED,1562,6369,24.53
1,2026-01,EXPIRED,1615,6369,25.36
2,2026-01,QUEUED,1553,6369,24.38
3,2026-01,SKIPPED,1639,6369,25.73
4,2026-02,CONTACTED,1409,5709,24.68
5,2026-02,EXPIRED,1473,5709,25.80
6,2026-02,QUEUED,1454,5709,25.47
7,2026-02,SKIPPED,1373,5709,24.05
8,2026-03,CONTACTED,1564,6290,24.86
9,2026-03,EXPIRED,1590,6290,25.28


In [9]:
# Calculate monthly recovery

payments = data["payments"].copy()
payments["event_at"] = pd.to_datetime(payments["event_at"])

payments["month"] = (
    payments["event_at"]
    .dt.to_period("M")
    .astype(str)
)

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
]

monthly_recovery = (
    successful_payments.groupby("month")["amount"]
    .sum()
    .reset_index(name="recovery_amount")
)

monthly_targeting = (
    targeting.groupby("month")["account_id"]
    .nunique()
    .reset_index(name="targeted_accounts")
)

counterfactual_data = monthly_recovery.merge(
    monthly_targeting,
    on="month",
    how="left"
)

counterfactual_data

,month,recovery_amount,targeted_accounts
0,2026-01,1.872291e+08,5732
1,2026-02,1.702796e+08,5160
2,2026-03,1.891903e+08,5666
3,2026-04,1.752289e+08,5585
4,2026-05,1.843355e+08,5800
5,2026-06,1.758534e+08,5535
6,2026-07,1.872478e+08,5666
7,2026-08,4.710970e+07,1566


In [10]:
# Calculate pre-change recovery per targeted account

pre_change = counterfactual_data[
    counterfactual_data["month"] <= "2026-06"
].copy()

pre_change["recovery_per_targeted_account"] = (
    pre_change["recovery_amount"]
    / pre_change["targeted_accounts"]
)

pre_change

,month,recovery_amount,targeted_accounts,recovery_per_targeted_account
0,2026-01,1.872291e+08,5732,32663.839426
1,2026-02,1.702796e+08,5160,32999.925382
2,2026-03,1.891903e+08,5666,33390.456509
3,2026-04,1.752289e+08,5585,31374.920353
4,2026-05,1.843355e+08,5800,31781.979941
5,2026-06,1.758534e+08,5535,31771.174320


In [11]:
# Calculate the pre-change baseline

baseline_recovery = pre_change["recovery_per_targeted_account"].mean()

print("Pre-change baseline recovery per targeted account:")
print(round(baseline_recovery, 2))

Pre-change baseline recovery per targeted account:
32330.38


In [13]:
# Calculate July counterfactual

july = counterfactual_data[
    counterfactual_data["month"] == "2026-07"
].iloc[0]

july_expected = (
    july["targeted_accounts"]
    * baseline_recovery
)

july_actual = july["recovery_amount"]

july_difference = july_actual - july_expected

july_change_pct = (
    july_difference
    / july_expected
    * 100
)

print("July actual recovery: ₹", round(july_actual, 2))
print("July expected recovery: ₹", round(july_expected, 2))
print("Difference: ₹", round(july_difference, 2))
print("Change vs counterfactual:", round(july_change_pct, 2), "%")

July actual recovery: ₹ 187247836.0
July expected recovery: ₹ 183183948.12
Difference: ₹ 4063887.88
Change vs counterfactual: 2.22 %


In [14]:
# Check August counterfactual

august = counterfactual_data[
    counterfactual_data["month"] == "2026-08"
].iloc[0]

august_expected = (
    august["targeted_accounts"]
    * baseline_recovery
)

august_actual = august["recovery_amount"]

august_difference = august_actual - august_expected

august_change_pct = (
    august_difference
    / august_expected
    * 100
)

print("August actual recovery: ₹", round(august_actual, 2))
print("August expected recovery: ₹", round(august_expected, 2))
print("Difference: ₹", round(august_difference, 2))
print("Change vs counterfactual:", round(august_change_pct, 2), "%")

August actual recovery: ₹ 47109695.31
August expected recovery: ₹ 50629379.24
Difference: ₹ -3519683.93
Change vs counterfactual: -6.95 %


In [15]:
# Check available channel data

print("Calls:", len(data["calls"]))
print("WhatsApp events:", len(data["whatsapp_events"]))
print("SMS events:", len(data["sms_events"]))
print("Field visits:", len(data["field_visits"]))

Calls: 90079
WhatsApp events: 60000
SMS events: 45000
Field visits: 25000


In [16]:
# Compare channel activity

channel_activity = pd.DataFrame({
    "channel": [
        "VOICE",
        "WHATSAPP",
        "SMS",
        "FIELD"
    ],
    "events": [
        len(data["calls"]),
        len(data["whatsapp_events"]),
        len(data["sms_events"]),
        len(data["field_visits"])
    ]
})

channel_activity

,channel,events
0,VOICE,90079
1,WHATSAPP,60000
2,SMS,45000
3,FIELD,25000


In [17]:
# Calculate channel-wise payment conversion

channel_tables = {
    "VOICE": data["calls"],
    "WHATSAPP": data["whatsapp_events"],
    "SMS": data["sms_events"],
    "FIELD": data["field_visits"]
}

channel_results = []

successful_accounts = set(
    data["payments"].loc[
        data["payments"]["payment_status"] == "SUCCESS",
        "account_id"
    ]
)

for channel, df in channel_tables.items():
    unique_accounts = df["account_id"].nunique()
    
    paid_accounts = len(
        set(df["account_id"]) & successful_accounts
    )
    
    conversion_rate = (
        paid_accounts / unique_accounts * 100
    )
    
    channel_results.append({
        "channel": channel,
        "unique_accounts": unique_accounts,
        "paid_accounts": paid_accounts,
        "conversion_rate": round(conversion_rate, 2)
    })

channel_results_df = pd.DataFrame(channel_results)

channel_results_df

,channel,unique_accounts,paid_accounts,conversion_rate
0,VOICE,28408,12596,44.34
1,WHATSAPP,25924,11506,44.38
2,SMS,23207,10310,44.43
3,FIELD,16908,7488,44.29


In [19]:
# Prepare voice events and successful payments

voice = data["calls"].copy()
payments = data["payments"].copy()

voice["voice_time"] = pd.to_datetime(voice["event_at"])
payments["payment_time"] = pd.to_datetime(payments["event_at"])

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

voice = voice.sort_values("voice_time")
successful_payments = successful_payments.sort_values("payment_time")

voice_payment = pd.merge_asof(
    successful_payments,
    voice[["account_id", "voice_time"]],
    left_on="payment_time",
    right_on="voice_time",
    by="account_id",
    direction="backward"
)

voice_payment["days_to_payment"] = (
    voice_payment["payment_time"]
    - voice_payment["voice_time"]
).dt.total_seconds() / (24 * 3600)

voice_payment = voice_payment[
    voice_payment["days_to_payment"] >= 0
]

voice_payment[["account_id", "days_to_payment"]].head()

,account_id,days_to_payment
188,ACC0029551,0.883634
206,ACC0029560,0.865370
254,ACC0014362,2.916678
280,ACC0010164,2.885405
310,ACC0005945,3.548414


In [20]:
# Summarize voice payment timing

voice_summary = {
    "channel": "VOICE",
    "payments_after_voice": len(voice_payment),
    "average_days_to_payment": round(
        voice_payment["days_to_payment"].mean(), 2
    ),
    "median_days_to_payment": round(
        voice_payment["days_to_payment"].median(), 2
    )
}

voice_summary

{'channel': 'VOICE',
 'payments_after_voice': 11933,
 'average_days_to_payment': np.float64(43.95),
 'median_days_to_payment': np.float64(33.13)}

In [21]:
# Prepare WhatsApp events and successful payments

whatsapp = data["whatsapp_events"].copy()
payments = data["payments"].copy()

whatsapp["whatsapp_time"] = pd.to_datetime(whatsapp["event_at"])
payments["payment_time"] = pd.to_datetime(payments["event_at"])

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

whatsapp = whatsapp.sort_values("whatsapp_time")
successful_payments = successful_payments.sort_values("payment_time")

whatsapp_payment = pd.merge_asof(
    successful_payments,
    whatsapp[["account_id", "whatsapp_time"]],
    left_on="payment_time",
    right_on="whatsapp_time",
    by="account_id",
    direction="backward"
)

whatsapp_payment["days_to_payment"] = (
    whatsapp_payment["payment_time"]
    - whatsapp_payment["whatsapp_time"]
).dt.total_seconds() / (24 * 3600)

whatsapp_payment = whatsapp_payment[
    whatsapp_payment["days_to_payment"] >= 0
]

whatsapp_payment[["account_id", "days_to_payment"]].head()

,account_id,days_to_payment
81,ACC0026010,0.511204
89,ACC0003049,1.130556
176,ACC0027477,0.639294
236,ACC0019819,2.175544
265,ACC0015077,2.055035


In [22]:
# Summarize WhatsApp payment timing

whatsapp_summary = {
    "channel": "WHATSAPP",
    "payments_after_whatsapp": len(whatsapp_payment),
    "average_days_to_payment": round(
        whatsapp_payment["days_to_payment"].mean(), 2
    ),
    "median_days_to_payment": round(
        whatsapp_payment["days_to_payment"].median(), 2
    )
}

whatsapp_summary

{'channel': 'WHATSAPP',
 'payments_after_whatsapp': 9994,
 'average_days_to_payment': np.float64(52.08),
 'median_days_to_payment': np.float64(40.65)}

In [23]:
# Prepare SMS events and successful payments

sms = data["sms_events"].copy()
payments = data["payments"].copy()

sms["sms_time"] = pd.to_datetime(sms["event_at"])
payments["payment_time"] = pd.to_datetime(payments["event_at"])

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

sms = sms.sort_values("sms_time")
successful_payments = successful_payments.sort_values("payment_time")

sms_payment = pd.merge_asof(
    successful_payments,
    sms[["account_id", "sms_time"]],
    left_on="payment_time",
    right_on="sms_time",
    by="account_id",
    direction="backward"
)

sms_payment["days_to_payment"] = (
    sms_payment["payment_time"]
    - sms_payment["sms_time"]
).dt.total_seconds() / (24 * 3600)

sms_payment = sms_payment[
    sms_payment["days_to_payment"] >= 0
]

sms_payment[["account_id", "days_to_payment"]].head()

,account_id,days_to_payment
65,ACC0010118,0.035162
81,ACC0026010,0.120567
101,ACC0027880,1.047604
124,ACC0008747,0.254711
146,ACC0025736,0.251713


In [24]:
# Summarize SMS payment timing

sms_summary = {
    "channel": "SMS",
    "payments_after_sms": len(sms_payment),
    "average_days_to_payment": round(
        sms_payment["days_to_payment"].mean(), 2
    ),
    "median_days_to_payment": round(
        sms_payment["days_to_payment"].median(), 2
    )
}

sms_summary

{'channel': 'SMS',
 'payments_after_sms': 8428,
 'average_days_to_payment': np.float64(56.14),
 'median_days_to_payment': np.float64(44.65)}

In [25]:
# Prepare field visits and successful payments

field = data["field_visits"].copy()
payments = data["payments"].copy()

field["field_time"] = pd.to_datetime(field["event_at"])
payments["payment_time"] = pd.to_datetime(payments["event_at"])

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

field = field.sort_values("field_time")
successful_payments = successful_payments.sort_values("payment_time")

field_payment = pd.merge_asof(
    successful_payments,
    field[["account_id", "field_time"]],
    left_on="payment_time",
    right_on="field_time",
    by="account_id",
    direction="backward"
)

field_payment["days_to_payment"] = (
    field_payment["payment_time"]
    - field_payment["field_time"]
).dt.total_seconds() / (24 * 3600)

field_payment = field_payment[
    field_payment["days_to_payment"] >= 0
]

field_payment[["account_id", "days_to_payment"]].head()

,account_id,days_to_payment
149,ACC0026314,0.519653
182,ACC0021217,2.175764
232,ACC0020136,1.360556
266,ACC0018434,1.025961
300,ACC0007655,3.794062


In [26]:
# Summarize field payment timing

field_summary = {
    "channel": "FIELD",
    "payments_after_field": len(field_payment),
    "average_days_to_payment": round(
        field_payment["days_to_payment"].mean(), 2
    ),
    "median_days_to_payment": round(
        field_payment["days_to_payment"].median(), 2
    )
}

field_summary

{'channel': 'FIELD',
 'payments_after_field': 5604,
 'average_days_to_payment': np.float64(62.96),
 'median_days_to_payment': np.float64(52.07)}

In [27]:
# Compare all channels

channel_summary = pd.DataFrame([
    voice_summary,
    whatsapp_summary,
    sms_summary,
    field_summary
])

channel_summary

,channel,payments_after_voice,average_days_to_payment,median_days_to_payment,payments_after_whatsapp,payments_after_sms,payments_after_field
0,VOICE,11933.0,43.95,33.13,NaN,NaN,NaN
1,WHATSAPP,NaN,52.08,40.65,9994.0,NaN,NaN
2,SMS,NaN,56.14,44.65,NaN,8428.0,NaN
3,FIELD,NaN,62.96,52.07,NaN,NaN,5604.0


In [28]:
# Create a simple channel performance score

channel_score = channel_summary[
    ["channel", "average_days_to_payment"]
].copy()

best_time = channel_score["average_days_to_payment"].min()

channel_score["speed_score"] = (
    best_time
    / channel_score["average_days_to_payment"]
    * 100
).round(2)

channel_score.sort_values(
    "speed_score",
    ascending=False
)

,channel,average_days_to_payment,speed_score
0,VOICE,43.95,100.00
1,WHATSAPP,52.08,84.39
2,SMS,56.14,78.29
3,FIELD,62.96,69.81


In [29]:
# Part 4 conclusion

print("Counterfactual analysis completed.")
print()
print("Key findings:")
print("- No clear mid-year change in campaign, channel, priority, or status mix.")
print("- July recovery was 2.22% above the pre-change baseline.")
print("- August was 6.95% below baseline, but the month is partial.")
print("- Voice had the fastest observed payment timing.")
print("- Channel timing alone does not prove causal impact.")
print()
print("Recommendation:")
print("Prioritize Voice for a controlled investment pilot.")
print("Use holdout testing before deploying the full ₹10 Cr.")

Counterfactual analysis completed.

Key findings:
- No clear mid-year change in campaign, channel, priority, or status mix.
- July recovery was 2.22% above the pre-change baseline.
- August was 6.95% below baseline, but the month is partial.
- Voice had the fastest observed payment timing.
- Channel timing alone does not prove causal impact.

Recommendation:
Prioritize Voice for a controlled investment pilot.
Use holdout testing before deploying the full ₹10 Cr.
